# 01 - Data Collection

This notebook fetches sumo wrestling data from sumo-api.com.

**Data Sources:**
- Primary: sumo-api.com API (1958-present)
- Fallback: Kaggle dataset (1983-2019)

**Outputs:**
- `rikishi.parquet` - All wrestler profiles
- `matches.parquet` - All bout records
- `kimarite.parquet` - Winning technique reference

In [ ]:
# Environment setup
import sys
import os

# Kaggle paths
INPUT_PATH = '/kaggle/input/sumo-data' if os.path.exists('/kaggle/input') else './data'
OUTPUT_PATH = '/kaggle/working' if os.path.exists('/kaggle/working') else './output'

# Add src to path for local development
if not os.path.exists('/kaggle/input'):
    sys.path.insert(0, '../src')
    os.makedirs('./data', exist_ok=True)
    os.makedirs('./output', exist_ok=True)

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
# Install dependencies if needed
!pip install -q requests pandas pyarrow

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
from pathlib import Path
from datetime import datetime
import json
from typing import Optional, List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

## API Client

In [ ]:
BASE_URL = "https://sumo-api.com/api"

# Rate limiting
REQUEST_DELAY = 0.5
MAX_RETRIES = 3
RETRY_DELAY = 5


def make_request(endpoint: str, params: Optional[Dict] = None) -> Optional[Dict]:
    """Make a request to sumo-api.com with retry logic."""
    url = f"{BASE_URL}{endpoint}"
    
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, params=params, timeout=30)
            
            if response.status_code == 200:
                time.sleep(REQUEST_DELAY)
                return response.json()
            elif response.status_code == 429:
                print(f"Rate limited, waiting {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)
                continue
            elif response.status_code == 404:
                return None
            else:
                print(f"Error {response.status_code} for {endpoint}")
                return None
                
        except requests.exceptions.RequestException as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
            time.sleep(RETRY_DELAY)
    
    return None

## Test API Connection

In [ ]:
# Test API connectivity
test_response = make_request("/rikishis", params={"limit": 1})

if test_response:
    print("API connection successful!")
    print(f"Total rikishi available: {test_response.get('total', 'unknown')}")
    API_AVAILABLE = True
else:
    print("API unavailable - will use Kaggle fallback dataset")
    API_AVAILABLE = False

## Fetch Kimarite Reference

In [ ]:
if API_AVAILABLE:
    kimarite_data = make_request("/kimarite")
    
    if kimarite_data and "records" in kimarite_data:
        kimarite_df = pd.DataFrame(kimarite_data["records"])
        print(f"Fetched {len(kimarite_df)} kimarite techniques")
        display(kimarite_df.head(10))
    else:
        kimarite_df = pd.DataFrame()

## Fetch All Rikishi (Wrestlers)

In [ ]:
def fetch_all_rikishi() -> pd.DataFrame:
    """Fetch all wrestler profiles with pagination."""
    print("Fetching all rikishi...")
    
    all_rikishi = []
    skip = 0
    limit = 1000
    
    while True:
        data = make_request("/rikishis", params={"skip": skip, "limit": limit})
        
        if not data or "records" not in data:
            break
        
        records = data["records"]
        if not records:
            break
        
        all_rikishi.extend(records)
        print(f"  Fetched {len(all_rikishi)} / {data.get('total', '?')} rikishi...")
        
        if len(records) < limit:
            break
        
        skip += limit
    
    print(f"Total rikishi fetched: {len(all_rikishi)}")
    return pd.DataFrame(all_rikishi)

In [ ]:
if API_AVAILABLE:
    rikishi_df = fetch_all_rikishi()
    print(f"\nRikishi DataFrame shape: {rikishi_df.shape}")
    print(f"\nColumns: {rikishi_df.columns.tolist()}")
    display(rikishi_df.head())

## Fetch Match History

We'll fetch matches for each rikishi. This is time-consuming but gives us the complete dataset.

In [ ]:
def fetch_rikishi_matches(rikishi_id: int) -> List[Dict]:
    """Fetch all matches for a specific wrestler."""
    all_matches = []
    skip = 0
    limit = 1000
    
    while True:
        data = make_request(f"/rikishi/{rikishi_id}/matches", 
                           params={"skip": skip, "limit": limit})
        
        if not data or "records" not in data:
            break
        
        records = data["records"]
        if not records:
            break
        
        all_matches.extend(records)
        
        if len(records) < limit:
            break
        
        skip += limit
    
    return all_matches

In [ ]:
def fetch_all_matches(rikishi_df: pd.DataFrame, 
                      max_rikishi: Optional[int] = None,
                      checkpoint_interval: int = 100) -> pd.DataFrame:
    """
    Fetch matches for all rikishi.
    
    Note: This deduplicates matches since each bout appears twice
    (once for each wrestler).
    """
    all_matches = []
    rikishi_ids = rikishi_df['id'].tolist()
    
    if max_rikishi:
        rikishi_ids = rikishi_ids[:max_rikishi]
    
    total = len(rikishi_ids)
    
    for i, rikishi_id in enumerate(rikishi_ids):
        matches = fetch_rikishi_matches(rikishi_id)
        all_matches.extend(matches)
        
        if (i + 1) % checkpoint_interval == 0:
            print(f"Progress: {i + 1}/{total} rikishi processed, {len(all_matches)} total match records")
    
    print(f"\nTotal match records (with duplicates): {len(all_matches)}")
    
    # Convert to DataFrame and deduplicate
    matches_df = pd.DataFrame(all_matches)
    
    if len(matches_df) > 0:
        # Create a unique match ID for deduplication
        # A bout is uniquely identified by basho, day, and the two wrestlers
        def create_bout_id(row):
            ids = sorted([str(row.get('eastId', '')), str(row.get('westId', ''))])
            return f"{row.get('bashoId', '')}_{row.get('day', '')}_{ids[0]}_{ids[1]}"
        
        matches_df['bout_id'] = matches_df.apply(create_bout_id, axis=1)
        matches_df = matches_df.drop_duplicates(subset=['bout_id'])
        print(f"After deduplication: {len(matches_df)} unique bouts")
    
    return matches_df

In [ ]:
# Fetch matches - this takes a while!
# For initial testing, limit to a subset
if API_AVAILABLE:
    # Filter to active divisions (those with recent basho participation)
    # Start with a smaller sample for testing
    
    # For full run, set max_rikishi=None
    # For testing, use max_rikishi=100
    matches_df = fetch_all_matches(rikishi_df, max_rikishi=None, checkpoint_interval=500)
    
    print(f"\nMatches DataFrame shape: {matches_df.shape}")
    print(f"\nColumns: {matches_df.columns.tolist()}")
    display(matches_df.head())

## Alternative: Load from Kaggle Dataset

If the API is unavailable or too slow, use the Kaggle fallback dataset.

In [ ]:
def load_kaggle_fallback():
    """
    Load data from Kaggle dataset if API is unavailable.
    
    Dataset: thedevastator/sumo-wrestling-matches-results-1985-2019
    """
    kaggle_path = '/kaggle/input/sumo-wrestling-matches-results-1985-2019'
    
    if os.path.exists(kaggle_path):
        # Load the Kaggle dataset
        files = os.listdir(kaggle_path)
        print(f"Available files: {files}")
        
        # Typically the dataset has a single CSV
        csv_files = [f for f in files if f.endswith('.csv')]
        if csv_files:
            df = pd.read_csv(os.path.join(kaggle_path, csv_files[0]))
            print(f"Loaded {len(df)} records from Kaggle dataset")
            return df
    
    print("Kaggle dataset not found")
    return None

In [ ]:
if not API_AVAILABLE:
    kaggle_df = load_kaggle_fallback()
    if kaggle_df is not None:
        display(kaggle_df.head())
        print(f"\nColumns: {kaggle_df.columns.tolist()}")

## Data Cleaning and Standardization

In [ ]:
def parse_banzuke_rank(rank_str: str) -> int:
    """
    Convert banzuke rank string to numeric value.
    Lower number = higher rank.
    """
    import re
    
    if not rank_str or pd.isna(rank_str):
        return 999
    
    rank_str = str(rank_str).strip().upper()
    
    # Rank prefixes and their base values
    rank_bases = {
        "Y": 0,      # Yokozuna
        "O": 10,     # Ozeki
        "S": 30,     # Sekiwake
        "K": 40,     # Komusubi
        "M": 50,     # Maegashira
        "J": 100,    # Juryo
    }
    
    match = re.match(r"([YOSKM]|J)(\d+)?([EW])?", rank_str)
    
    if not match:
        return 999
    
    rank_letter = match.group(1)
    rank_num = int(match.group(2)) if match.group(2) else 1
    direction = match.group(3) if match.group(3) else "E"
    
    base = rank_bases.get(rank_letter, 999)
    numeric = base + (rank_num - 1) * 2
    
    if direction == "W":
        numeric += 1
    
    return numeric

In [ ]:
# Kimarite categories
KIMARITE_CATEGORIES = {
    "push": [
        "oshidashi", "tsukidashi", "oshitaoshi", "tsukiotoshi",
        "tsukitaoshi", "okuridashi", "abisetaoshi"
    ],
    "grapple": [
        "yorikiri", "uwatenage", "shitatenage", "sukuinage", "kotenage",
        "kubinage", "yoritaoshi", "uwatedashinage", "shitatedashinage",
        "kakenage", "kirikaeshi", "tsukaminage", "tsuridashi", "tsuriotoshi",
        "utchari", "sotogake", "uchigake", "kimedashi", "kimekiri",
        "katasukashi", "okurinage", "okuritaoshi", "okurihineri",
        "okuritsuridashi", "amiuchi", "sabaori", "waridashi", "makiotoshi",
        "uwatehineri", "shitatehineri"
    ],
    "evasion": [
        "hatakikomi", "hikiotoshi", "hikkake", "ketaguri", "kekaeshi",
        "ashitori", "tsumadori", "chongake", "kawazugake", "komatasukui",
        "tottari", "izori", "shumokuzori", "tasukizori", "nichonage"
    ]
}


def categorize_kimarite(kimarite: str) -> str:
    """Categorize a kimarite into push/grapple/evasion."""
    if not kimarite or pd.isna(kimarite):
        return "unknown"
    
    kimarite_lower = str(kimarite).lower().strip()
    
    for category, techniques in KIMARITE_CATEGORIES.items():
        if kimarite_lower in techniques:
            return category
    
    return "grapple"  # Default

In [ ]:
def clean_matches_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and standardize match data."""
    df = df.copy()
    
    # Parse basho date
    if 'bashoId' in df.columns:
        df['basho_year'] = df['bashoId'].astype(str).str[:4].astype(int)
        df['basho_month'] = df['bashoId'].astype(str).str[4:6].astype(int)
    
    # Parse ranks to numeric
    if 'eastRank' in df.columns:
        df['east_rank_numeric'] = df['eastRank'].apply(parse_banzuke_rank)
    if 'westRank' in df.columns:
        df['west_rank_numeric'] = df['westRank'].apply(parse_banzuke_rank)
    
    # Categorize kimarite
    if 'kimarite' in df.columns:
        df['kimarite_category'] = df['kimarite'].apply(categorize_kimarite)
    
    # Create winner columns
    if 'winnerId' in df.columns and 'eastId' in df.columns:
        df['east_won'] = (df['winnerId'] == df['eastId']).astype(int)
        df['west_won'] = (df['winnerId'] == df['westId']).astype(int)
    
    return df

In [ ]:
if API_AVAILABLE and len(matches_df) > 0:
    matches_df = clean_matches_data(matches_df)
    print(f"Cleaned matches DataFrame shape: {matches_df.shape}")
    display(matches_df.head())

## Data Summary

In [ ]:
if API_AVAILABLE and len(matches_df) > 0:
    print("=== Data Summary ===")
    print(f"\nTotal unique bouts: {len(matches_df):,}")
    print(f"Total unique rikishi: {len(rikishi_df):,}")
    
    if 'basho_year' in matches_df.columns:
        print(f"\nYear range: {matches_df['basho_year'].min()} - {matches_df['basho_year'].max()}")
    
    if 'kimarite' in matches_df.columns:
        print(f"\nTop 10 kimarite:")
        print(matches_df['kimarite'].value_counts().head(10))
    
    if 'kimarite_category' in matches_df.columns:
        print(f"\nKimarite by category:")
        print(matches_df['kimarite_category'].value_counts())

## Save Data

In [ ]:
# Save to parquet format
if API_AVAILABLE:
    if len(rikishi_df) > 0:
        rikishi_df.to_parquet(f"{OUTPUT_PATH}/rikishi.parquet", index=False)
        print(f"Saved {len(rikishi_df)} rikishi to {OUTPUT_PATH}/rikishi.parquet")
    
    if len(matches_df) > 0:
        matches_df.to_parquet(f"{OUTPUT_PATH}/matches.parquet", index=False)
        print(f"Saved {len(matches_df)} matches to {OUTPUT_PATH}/matches.parquet")
    
    if len(kimarite_df) > 0:
        kimarite_df.to_parquet(f"{OUTPUT_PATH}/kimarite.parquet", index=False)
        print(f"Saved {len(kimarite_df)} kimarite to {OUTPUT_PATH}/kimarite.parquet")

In [ ]:
# Verify saved files
import os

print(f"\nFiles in {OUTPUT_PATH}:")
if os.path.exists(OUTPUT_PATH):
    for f in os.listdir(OUTPUT_PATH):
        filepath = os.path.join(OUTPUT_PATH, f)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {f}: {size_mb:.2f} MB")

## Next Steps

The data is now ready for the next notebook:
- `02_rating_systems.ipynb` - Compute ELO and Glicko-2 ratings